In [1]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
from xgboost import XGBRegressor


In [2]:
df = pd.read_csv('../data/data_for_assignment/cleaned_flight_data.csv')
df.head()

,Total_Stops,Price,day_of_week,is_weekend,Date,Month,year,Arrival_hour,Arrival_Minute,Dep_hour,...,Source_Chennai,Source_Delhi,Source_Kolkata,Source_Mumbai,Destination_Banglore,Destination_Cochin,Destination_Delhi,Destination_Hyderabad,Destination_Kolkata,Destination_New Delhi
0,0,3897,6,1,24,3,2019,1,10,22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,7662,2,0,1,5,2019,13,15,5,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,2,13882,6,1,9,6,2019,4,25,9,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1,6218,6,1,12,5,2019,23,30,18,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,1,13302,4,0,1,3,2019,21,35,16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
#save models
model_dir = '..\\models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

In [4]:
#split data into training and testing sets
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Samples of all: {len(df)}")
print(f"Samples of training set: {len(X_train)}")
print(f"Samples of testing set: {len(X_test)}")

Samples of all: 10589
Samples of training set: 8471
Samples of testing set: 2118


In [5]:
#Algorithm selection and training
#train random forest regressor (RANDOM FOREST REGRESSOR)
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
#train the model
rf_model.fit(X_train, y_train)
#predict on the test set
y_pred = rf_model.predict(X_test)

Training Random Forest Regressor...


In [6]:
#XGBoost Regressor
print("Training XGBoost Regressor...")
xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

Training XGBoost Regressor...


In [7]:
#function to calculate evaluation metrics
def evaluate_model(y_true, y_pred, model_name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    print(f"{model_name} Evaluation:")
    print(f"R-squared: {r2:.4f}")
    print(f"Mean Absolute Error: {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print("-" * 30)
    return r2

In [8]:
#Compare models
r2_rf = evaluate_model(y_test, y_pred, "Random Forest Regressor")
r2_xgb = evaluate_model(y_test, y_pred_xgb, "XGBoost Regressor")

Random Forest Regressor Evaluation:
R-squared: 0.8095
Mean Absolute Error: 1118.71
Root Mean Squared Error (RMSE): 1754.00
------------------------------
XGBoost Regressor Evaluation:
R-squared: 0.8421
Mean Absolute Error: 1071.32
Root Mean Squared Error (RMSE): 1597.19
------------------------------


In [9]:
#save models
rf_path = os.path.join(model_dir, 'rf_flight_price_model.pkl')
xgb_path = os.path.join(model_dir, 'xg_flight_price_model.json')
joblib.dump(rf_model, rf_path)
xgb_model.save_model(xgb_path)

print(f"Successfully saved Random Forest model to {rf_path}")
print(f"Successfully saved XGBoost model to {xgb_path}")

Successfully saved Random Forest model to ..\models\rf_flight_price_model.pkl
Successfully saved XGBoost model to ..\models\xg_flight_price_model.json


In [10]:
#chose the best model based on R-squared
best_model = "Random Forest" if r2_rf > r2_xgb else "XGBoost"
print(f"The best model is: {best_model}")

The best model is: XGBoost


In [11]:
#change parameters of the best model and retrain
from sklearn.model_selection import RandomizedSearchCV
print("Performing hyperparameter tuning for the best model XGBoost Regressor...")
xgb_params = {
    'n_estimators': [500, 1000],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [8, 10, 12],
    'gamma': [0, 0.1, 0.2] 
}

xgb_cv = RandomizedSearchCV(XGBRegressor(random_state=42), xgb_params, n_iter=20, cv=3, scoring='r2', n_jobs=-1)
xgb_cv.fit(X_train, y_train)

#chose the best hyperparameters and retrain the model
best_xgb_model = xgb_cv.best_estimator_
y_pred_best_xgb = best_xgb_model.predict(X_test)

print("Result after hyperparameter tuning:")
evaluate_model(y_test, y_pred_best_xgb, "Optimized XGBoost")

Performing hyperparameter tuning for the best model XGBoost Regressor...
Result after hyperparameter tuning:
Optimized XGBoost Evaluation:
R-squared: 0.8423
Mean Absolute Error: 1122.08
Root Mean Squared Error (RMSE): 1596.05
------------------------------


0.8422954082489014

In [12]:
#save the optimized model
best_xgb_path = os.path.join(model_dir, 'best_flight_price_model_optimized.json')
best_xgb_model.save_model(best_xgb_path)
print(f"Successfully saved the optimized XGBoost model to {best_xgb_path}")

Successfully saved the optimized XGBoost model to ..\models\best_flight_price_model_optimized.json
